In [6]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

## 1.把原始数据文件，整理成一个“audio-MIDI 配对清单”。

In [7]:

# =========================
# 1. 路径设置
# =========================

PROJECT_ROOT = Path(r"D:\organ-amt-generalization")

ORGAN_META_PATH = PROJECT_ROOT / r"data\raw\organ\ex01_bachorgan\metadata.csv"

PIANO_AUDIO_DIR = PROJECT_ROOT / r"data\raw\piano\maestro_v3_2018_10\audio"
PIANO_MIDI_DIR  = PROJECT_ROOT / r"data\raw\piano\maestro_v3_2018_10\midi"
PIANO_META_PATH = PROJECT_ROOT / r"data\raw\piano\maestro_v3_2018_10\maestro-v3.0.0-2018-10.csv"

PROCESSED_DIR = PROJECT_ROOT / r"data\processed\ex01_data_check"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# 2. 检查路径
# =========================

paths = {
    "PROJECT_ROOT": PROJECT_ROOT,
    "ORGAN_META_PATH": ORGAN_META_PATH,
    "PIANO_AUDIO_DIR": PIANO_AUDIO_DIR,
    "PIANO_MIDI_DIR": PIANO_MIDI_DIR,
    "PIANO_META_PATH": PIANO_META_PATH,
    "PROCESSED_DIR": PROCESSED_DIR,
}

print("Path check:")
for name, path in paths.items():
    print(f"{name:20s} {str(path.exists()):5s} | {path}")

missing_required_paths = [
    name for name, path in paths.items()
    if name != "PROCESSED_DIR" and not path.exists()
]

if missing_required_paths:
    raise FileNotFoundError(f"以下路径不存在，请先检查路径: {missing_required_paths}")

# =========================
# 3. 读取 metadata
# =========================

organ_meta = pd.read_csv(ORGAN_META_PATH)
piano_meta = pd.read_csv(PIANO_META_PATH)

print("\nMetadata shape:")
print("organ_meta:", organ_meta.shape)
print("piano_meta:", piano_meta.shape)

print("\nOrgan columns:")
print(organ_meta.columns.tolist())

print("\nPiano columns:")
print(piano_meta.columns.tolist())

# =========================
# 4. 工具函数
# =========================

AUDIO_EXTS = {".wav", ".flac", ".mp3", ".ogg", ".m4a"}
MIDI_EXTS = {".mid", ".midi"}


def scan_files(root_dir, exts):
    root_dir = Path(root_dir)
    files = []

    for path in root_dir.rglob("*"):
        if path.is_file() and path.suffix.lower() in exts:
            files.append(path)

    return sorted(files)


def build_file_index(root_dir, exts):
    files = scan_files(root_dir, exts)

    by_name = {}
    by_stem = {}

    for path in files:
        by_name.setdefault(path.name, path)
        by_stem.setdefault(path.stem, path)

    return {
        "files": files,
        "by_name": by_name,
        "by_stem": by_stem,
    }


def resolve_metadata_path(project_root, path_value):
    if pd.isna(path_value):
        return None

    path_str = str(path_value).strip()
    path = Path(path_str)

    if path.is_absolute():
        return path

    return project_root / path


def resolve_maestro_file(base_dir, filename, file_index):
    if pd.isna(filename):
        return None

    filename_path = Path(str(filename).strip())

    candidates = [
        Path(base_dir) / filename_path,
        Path(base_dir) / filename_path.name,
    ]

    for path in candidates:
        if path.exists():
            return path

    if filename_path.name in file_index["by_name"]:
        return file_index["by_name"][filename_path.name]

    if filename_path.stem in file_index["by_stem"]:
        return file_index["by_stem"][filename_path.stem]

    return candidates[0]

# =========================
# 5. 构造 organ 配对表
# =========================

def build_organ_pairs(organ_meta, project_root):
    required_cols = ["id", "audio_path", "midi_path"]
    missing_cols = [col for col in required_cols if col not in organ_meta.columns]

    if missing_cols:
        raise ValueError(f"organ metadata 缺少必要字段: {missing_cols}")

    rows = []

    for _, row in organ_meta.iterrows():
        audio_path = resolve_metadata_path(project_root, row["audio_path"])
        midi_path = resolve_metadata_path(project_root, row["midi_path"])

        has_audio = audio_path is not None and audio_path.exists()
        has_midi = midi_path is not None and midi_path.exists()

        rows.append({
            "piece_id": row["id"],
            "instrument": "organ",
            "source": row.get("dataset", "ex01_bachorgan"),
            "domain": row.get("domain", "organ"),
            "audio_path": str(audio_path) if audio_path is not None else None,
            "midi_path": str(midi_path) if midi_path is not None else None,
            "has_audio": has_audio,
            "has_midi": has_midi,
            "metadata_duration": row.get("duration", None),
            "sample_rate": row.get("sample_rate", None),
            "channels": row.get("channels", None),
            "split": None,
            "composer": None,
            "title": None,
            "soundfont": row.get("soundfont", None),
            "synthesizer": row.get("synthesizer", None),
            "maestro_year": None,
            "maestro_audio_filename": None,
            "maestro_midi_filename": None,
        })

    return pd.DataFrame(rows)


organ_pairs = build_organ_pairs(
    organ_meta=organ_meta,
    project_root=PROJECT_ROOT,
)

# =========================
# 6. 构造 piano 配对表
# =========================

piano_audio_index = build_file_index(PIANO_AUDIO_DIR, AUDIO_EXTS)
piano_midi_index = build_file_index(PIANO_MIDI_DIR, MIDI_EXTS)

print("\nScanned piano files:")
print("piano audio files:", len(piano_audio_index["files"]))
print("piano midi files :", len(piano_midi_index["files"]))


def build_piano_pairs(piano_meta, audio_dir, midi_dir, audio_index, midi_index):
    required_cols = ["audio_filename", "midi_filename"]
    missing_cols = [col for col in required_cols if col not in piano_meta.columns]

    if missing_cols:
        raise ValueError(f"piano metadata 缺少必要字段: {missing_cols}")

    rows = []

    for _, row in piano_meta.iterrows():
        audio_path = resolve_maestro_file(
            base_dir=audio_dir,
            filename=row["audio_filename"],
            file_index=audio_index,
        )

        midi_path = resolve_maestro_file(
            base_dir=midi_dir,
            filename=row["midi_filename"],
            file_index=midi_index,
        )

        has_audio = audio_path is not None and audio_path.exists()
        has_midi = midi_path is not None and midi_path.exists()

        piece_id = Path(str(row["audio_filename"])).stem

        rows.append({
            "piece_id": piece_id,
            "instrument": "piano",
            "source": "maestro_v3_2018_10",
            "domain": "piano",
            "audio_path": str(audio_path) if audio_path is not None else None,
            "midi_path": str(midi_path) if midi_path is not None else None,
            "has_audio": has_audio,
            "has_midi": has_midi,
            "metadata_duration": row.get("duration", None),
            "sample_rate": None,
            "channels": None,
            "split": row.get("split", None),
            "composer": row.get("canonical_composer", None),
            "title": row.get("canonical_title", None),
            "soundfont": None,
            "synthesizer": None,
            "maestro_year": row.get("year", None),
            "maestro_audio_filename": row.get("audio_filename", None),
            "maestro_midi_filename": row.get("midi_filename", None),
        })

    return pd.DataFrame(rows)


piano_pairs = build_piano_pairs(
    piano_meta=piano_meta,
    audio_dir=PIANO_AUDIO_DIR,
    midi_dir=PIANO_MIDI_DIR,
    audio_index=piano_audio_index,
    midi_index=piano_midi_index,
)

# =========================
# 7. 合并与统计
# =========================

file_pairs = pd.concat(
    [organ_pairs, piano_pairs],
    ignore_index=True,
)

paired_files = file_pairs[
    file_pairs["has_audio"] & file_pairs["has_midi"]
].copy()

missing_files = file_pairs[
    ~(file_pairs["has_audio"] & file_pairs["has_midi"])
].copy()

summary = (
    file_pairs
    .groupby(["instrument", "has_audio", "has_midi"])
    .size()
    .reset_index(name="count")
)

valid_summary = (
    paired_files
    .groupby("instrument")
    .size()
    .reset_index(name="paired_count")
)

print("\nPair status:")
display(summary)

print("\nValid paired records:")
display(valid_summary)

print("\nTotal records:")
print("all records    :", len(file_pairs))
print("paired records :", len(paired_files))
print("missing records:", len(missing_files))

print("\nFile pairs head:")
display(file_pairs.head())

print("\nMissing files:")
display(missing_files)

# =========================
# 8. 保存结果
# =========================

file_pairs_path = PROCESSED_DIR / "file_pairs.csv"
paired_files_path = PROCESSED_DIR / "paired_files.csv"
missing_files_path = PROCESSED_DIR / "missing_files.csv"
summary_path = PROCESSED_DIR / "pair_summary.csv"
valid_summary_path = PROCESSED_DIR / "valid_pair_summary.csv"

file_pairs.to_csv(file_pairs_path, index=False, encoding="utf-8-sig")
paired_files.to_csv(paired_files_path, index=False, encoding="utf-8-sig")
missing_files.to_csv(missing_files_path, index=False, encoding="utf-8-sig")
summary.to_csv(summary_path, index=False, encoding="utf-8-sig")
valid_summary.to_csv(valid_summary_path, index=False, encoding="utf-8-sig")

print("\nSaved files:")
print(file_pairs_path)
print(paired_files_path)
print(missing_files_path)
print(summary_path)
print(valid_summary_path)

# =========================
# 9. 保存后复查
# =========================

check_file_pairs = pd.read_csv(file_pairs_path)
check_paired_files = pd.read_csv(paired_files_path)
check_missing_files = pd.read_csv(missing_files_path)

print("\nReload check:")
print("file_pairs.csv shape   :", check_file_pairs.shape)
print("paired_files.csv shape :", check_paired_files.shape)
print("missing_files.csv shape:", check_missing_files.shape)

Path check:
PROJECT_ROOT         True  | D:\organ-amt-generalization
ORGAN_META_PATH      True  | D:\organ-amt-generalization\data\raw\organ\ex01_bachorgan\metadata.csv
PIANO_AUDIO_DIR      True  | D:\organ-amt-generalization\data\raw\piano\maestro_v3_2018_10\audio
PIANO_MIDI_DIR       True  | D:\organ-amt-generalization\data\raw\piano\maestro_v3_2018_10\midi
PIANO_META_PATH      True  | D:\organ-amt-generalization\data\raw\piano\maestro_v3_2018_10\maestro-v3.0.0-2018-10.csv
PROCESSED_DIR        True  | D:\organ-amt-generalization\data\processed\ex01_data_check

Metadata shape:
organ_meta: (15, 10)
piano_meta: (10, 7)

Organ columns:
['id', 'audio_path', 'midi_path', 'dataset', 'domain', 'soundfont', 'synthesizer', 'sample_rate', 'channels', 'duration']

Piano columns:
['canonical_composer', 'canonical_title', 'split', 'year', 'midi_filename', 'audio_filename', 'duration']

Scanned piano files:
piano audio files: 10
piano midi files : 10

Pair status:


,instrument,has_audio,has_midi,count
0,organ,True,True,15
1,piano,True,True,10



Valid paired records:


,instrument,paired_count
0,organ,15
1,piano,10



Total records:
all records    : 25
paired records : 25
missing records: 0

File pairs head:


,piece_id,instrument,source,domain,audio_path,midi_path,has_audio,has_midi,metadata_duration,sample_rate,channels,split,composer,title,soundfont,synthesizer,maestro_year,maestro_audio_filename,maestro_midi_filename
0,faf_gm11,organ,ex01_bachorgan,organ,D:\organ-amt-generalization\data\raw\organ\ex0...,D:\organ-amt-generalization\data\raw\organ\ex0...,True,True,350.064036,44100,2,None,None,None,data/raw/soundfonts/Pipe Organ Samples v1.1.sf2,fluidsynth,None,None,None
1,faf_gm12,organ,ex01_bachorgan,organ,D:\organ-amt-generalization\data\raw\organ\ex0...,D:\organ-amt-generalization\data\raw\organ\ex0...,True,True,399.139410,44100,2,None,None,None,data/raw/soundfonts/Pipe Organ Samples v1.1.sf2,fluidsynth,None,None,None
2,fafxgm11,organ,ex01_bachorgan,organ,D:\organ-amt-generalization\data\raw\organ\ex0...,D:\organ-amt-generalization\data\raw\organ\ex0...,True,True,350.064036,44100,2,None,None,None,data/raw/soundfonts/Pipe Organ Samples v1.1.sf2,fluidsynth,None,None,None
3,fafxgm12,organ,ex01_bachorgan,organ,D:\organ-amt-generalization\data\raw\organ\ex0...,D:\organ-amt-generalization\data\raw\organ\ex0...,True,True,406.639456,44100,2,None,None,None,data/raw/soundfonts/Pipe Organ Samples v1.1.sf2,fluidsynth,None,None,None
4,fai_hm11,organ,ex01_bachorgan,organ,D:\organ-amt-generalization\data\raw\organ\ex0...,D:\organ-amt-generalization\data\raw\organ\ex0...,True,True,94.946395,44100,2,None,None,None,data/raw/soundfonts/Pipe Organ Samples v1.1.sf2,fluidsynth,None,None,None



Missing files:


,piece_id,instrument,source,domain,audio_path,midi_path,has_audio,has_midi,metadata_duration,sample_rate,channels,split,composer,title,soundfont,synthesizer,maestro_year,maestro_audio_filename,maestro_midi_filename



Saved files:
D:\organ-amt-generalization\data\processed\ex01_data_check\file_pairs.csv
D:\organ-amt-generalization\data\processed\ex01_data_check\paired_files.csv
D:\organ-amt-generalization\data\processed\ex01_data_check\missing_files.csv
D:\organ-amt-generalization\data\processed\ex01_data_check\pair_summary.csv
D:\organ-amt-generalization\data\processed\ex01_data_check\valid_pair_summary.csv

Reload check:
file_pairs.csv shape   : (25, 19)
paired_files.csv shape : (25, 19)
missing_files.csv shape: (0, 19)


## 2内容检查 
能否读取 是不是静音 是否为空文件 采样率 时长会不会差很多

In [8]:


# =========================
# 1. 路径设置
# =========================

PROJECT_ROOT = Path(r"D:\organ-amt-generalization")
PROCESSED_DIR = PROJECT_ROOT / r"data\processed\ex01_data_check"

PAIRED_FILES_PATH = PROCESSED_DIR / "paired_files.csv"

CONTENT_CHECK_PATH = PROCESSED_DIR / "content_check.csv"
VALID_SAMPLES_PATH = PROCESSED_DIR / "valid_samples.csv"
INVALID_SAMPLES_PATH = PROCESSED_DIR / "invalid_samples.csv"
CONTENT_SUMMARY_PATH = PROCESSED_DIR / "content_check_summary.csv"
PROBLEM_SUMMARY_PATH = PROCESSED_DIR / "problem_summary.csv"

print("PROCESSED_DIR     :", PROCESSED_DIR)
print("PAIRED_FILES_PATH :", PAIRED_FILES_PATH)

if not PAIRED_FILES_PATH.exists():
    raise FileNotFoundError(f"找不到 paired_files.csv，请先运行文件检查单元格: {PAIRED_FILES_PATH}")

# =========================
# 2. 导入音频和 MIDI 读取库
# =========================

try:
    import soundfile as sf
except ImportError:
    raise ImportError(
        "缺少 soundfile，请先在当前 conda 环境中运行：pip install soundfile"
    )

try:
    import pretty_midi
except ImportError:
    raise ImportError(
        "缺少 pretty_midi，请先在当前 conda 环境中运行：pip install pretty_midi"
    )

# =========================
# 3. 读取文件配对表
# =========================

paired_files = pd.read_csv(PAIRED_FILES_PATH)

print("\npaired_files shape:", paired_files.shape)
display(paired_files.head())

required_cols = ["piece_id", "instrument", "audio_path", "midi_path", "has_audio", "has_midi"]
missing_cols = [col for col in required_cols if col not in paired_files.columns]

if missing_cols:
    raise ValueError(f"paired_files.csv 缺少必要字段: {missing_cols}")

# =========================
# 4. 检查参数
# =========================

SILENT_RMS_THRESHOLD = 1e-4
CLIP_AMPLITUDE_THRESHOLD = 0.999
CLIP_RATIO_THRESHOLD = 0.001
DURATION_DIFF_THRESHOLD = 3.0

print("\nCheck thresholds:")
print("SILENT_RMS_THRESHOLD   :", SILENT_RMS_THRESHOLD)
print("CLIP_AMPLITUDE_THRESHOLD:", CLIP_AMPLITUDE_THRESHOLD)
print("CLIP_RATIO_THRESHOLD   :", CLIP_RATIO_THRESHOLD)
print("DURATION_DIFF_THRESHOLD:", DURATION_DIFF_THRESHOLD, "sec")

# =========================
# 5. Audio 检查函数
# =========================

def check_audio(audio_path):
    result = {
        "audio_readable": False,
        "audio_error": None,
        "sample_rate_checked": np.nan,
        "num_channels_checked": np.nan,
        "num_samples": np.nan,
        "audio_duration_sec": np.nan,
        "max_amplitude": np.nan,
        "rms_amplitude": np.nan,
        "clip_ratio": np.nan,
        "is_silent": False,
        "is_clipped": False,
    }

    try:
        audio_path = Path(audio_path)

        if not audio_path.exists():
            result["audio_error"] = "audio_file_not_found"
            return result

        audio, sr = sf.read(audio_path, always_2d=True)

        num_samples = audio.shape[0]
        num_channels = audio.shape[1]

        if num_samples == 0:
            result.update({
                "audio_readable": True,
                "sample_rate_checked": sr,
                "num_channels_checked": num_channels,
                "num_samples": 0,
                "audio_duration_sec": 0.0,
                "max_amplitude": 0.0,
                "rms_amplitude": 0.0,
                "clip_ratio": 0.0,
                "is_silent": True,
                "is_clipped": False,
            })
            return result

        abs_audio = np.abs(audio)
        max_amp = float(np.max(abs_audio))
        rms_amp = float(np.sqrt(np.mean(audio ** 2)))
        clip_ratio = float(np.mean(abs_audio >= CLIP_AMPLITUDE_THRESHOLD))
        duration = float(num_samples / sr)

        result.update({
            "audio_readable": True,
            "sample_rate_checked": int(sr),
            "num_channels_checked": int(num_channels),
            "num_samples": int(num_samples),
            "audio_duration_sec": duration,
            "max_amplitude": max_amp,
            "rms_amplitude": rms_amp,
            "clip_ratio": clip_ratio,
            "is_silent": bool(rms_amp < SILENT_RMS_THRESHOLD),
            "is_clipped": bool(clip_ratio > CLIP_RATIO_THRESHOLD),
        })

        return result

    except Exception as e:
        result["audio_error"] = repr(e)
        return result

# =========================
# 6. MIDI 检查函数
# =========================

def check_midi(midi_path):
    result = {
        "midi_readable": False,
        "midi_error": None,
        "num_notes": np.nan,
        "midi_duration_sec": np.nan,
        "min_pitch": np.nan,
        "max_pitch": np.nan,
        "mean_pitch": np.nan,
        "num_instruments": np.nan,
        "num_nonempty_instruments": np.nan,
        "num_programs": np.nan,
    }

    try:
        midi_path = Path(midi_path)

        if not midi_path.exists():
            result["midi_error"] = "midi_file_not_found"
            return result

        pm = pretty_midi.PrettyMIDI(str(midi_path))

        notes = []
        programs = []

        for inst in pm.instruments:
            programs.append(inst.program)
            for note in inst.notes:
                notes.append(note)

        num_notes = len(notes)
        num_instruments = len(pm.instruments)
        num_nonempty_instruments = sum(len(inst.notes) > 0 for inst in pm.instruments)
        num_programs = len(set(programs))

        if num_notes > 0:
            pitches = np.array([note.pitch for note in notes], dtype=float)
            min_pitch = float(np.min(pitches))
            max_pitch = float(np.max(pitches))
            mean_pitch = float(np.mean(pitches))
        else:
            min_pitch = np.nan
            max_pitch = np.nan
            mean_pitch = np.nan

        midi_duration = float(pm.get_end_time())

        result.update({
            "midi_readable": True,
            "num_notes": int(num_notes),
            "midi_duration_sec": midi_duration,
            "min_pitch": min_pitch,
            "max_pitch": max_pitch,
            "mean_pitch": mean_pitch,
            "num_instruments": int(num_instruments),
            "num_nonempty_instruments": int(num_nonempty_instruments),
            "num_programs": int(num_programs),
        })

        return result

    except Exception as e:
        result["midi_error"] = repr(e)
        return result

# =========================
# 7. 有效性判断函数
# =========================

def judge_sample(row):
    problems = []

    if not bool(row.get("has_audio", False)):
        problems.append("missing_audio")

    if not bool(row.get("has_midi", False)):
        problems.append("missing_midi")

    if not bool(row.get("audio_readable", False)):
        problems.append("audio_not_readable")

    if not bool(row.get("midi_readable", False)):
        problems.append("midi_not_readable")

    if bool(row.get("is_silent", False)):
        problems.append("silent_audio")

    if bool(row.get("is_clipped", False)):
        problems.append("clipped_audio")

    num_notes = row.get("num_notes", np.nan)
    if pd.isna(num_notes) or num_notes <= 0:
        problems.append("empty_midi")

    duration_match = row.get("duration_match", False)
    if not bool(duration_match):
        problems.append("duration_mismatch")

    is_valid = len(problems) == 0
    problem_type = "valid" if is_valid else ";".join(problems)

    return pd.Series({
        "is_valid": is_valid,
        "problem_type": problem_type,
    })

# =========================
# 8. 执行内容检查
# =========================

rows = []

for idx, row in paired_files.iterrows():
    audio_path = row["audio_path"]
    midi_path = row["midi_path"]

    audio_info = check_audio(audio_path)
    midi_info = check_midi(midi_path)

    item = row.to_dict()
    item.update(audio_info)
    item.update(midi_info)

    audio_duration = item.get("audio_duration_sec", np.nan)
    midi_duration = item.get("midi_duration_sec", np.nan)

    if pd.notna(audio_duration) and pd.notna(midi_duration):
        duration_diff = abs(float(audio_duration) - float(midi_duration))
        duration_match = duration_diff <= DURATION_DIFF_THRESHOLD
    else:
        duration_diff = np.nan
        duration_match = False

    item["duration_diff_sec"] = duration_diff
    item["duration_match"] = bool(duration_match)

    rows.append(item)

    if (idx + 1) % 20 == 0 or (idx + 1) == len(paired_files):
        print(f"checked {idx + 1}/{len(paired_files)}")

content_check = pd.DataFrame(rows)

judge_result = content_check.apply(judge_sample, axis=1)
content_check = pd.concat([content_check, judge_result], axis=1)

valid_samples = content_check[content_check["is_valid"]].copy()
invalid_samples = content_check[~content_check["is_valid"]].copy()

# =========================
# 9. 汇总统计
# =========================

content_summary = (
    content_check
    .groupby("instrument")
    .agg(
        total=("piece_id", "count"),
        valid=("is_valid", "sum"),
        audio_readable=("audio_readable", "sum"),
        midi_readable=("midi_readable", "sum"),
        mean_audio_duration_sec=("audio_duration_sec", "mean"),
        mean_midi_duration_sec=("midi_duration_sec", "mean"),
        mean_duration_diff_sec=("duration_diff_sec", "mean"),
        max_duration_diff_sec=("duration_diff_sec", "max"),
        mean_num_notes=("num_notes", "mean"),
        min_num_notes=("num_notes", "min"),
        max_num_notes=("num_notes", "max"),
    )
    .reset_index()
)

content_summary["invalid"] = content_summary["total"] - content_summary["valid"]
content_summary["valid_rate"] = content_summary["valid"] / content_summary["total"]

problem_rows = []

for _, row in invalid_samples.iterrows():
    problems = str(row["problem_type"]).split(";")
    for problem in problems:
        problem_rows.append({
            "piece_id": row["piece_id"],
            "instrument": row["instrument"],
            "problem_type": problem,
        })

if len(problem_rows) > 0:
    problem_detail = pd.DataFrame(problem_rows)
    problem_summary = (
        problem_detail
        .groupby(["instrument", "problem_type"])
        .size()
        .reset_index(name="count")
        .sort_values(["instrument", "count"], ascending=[True, False])
    )
else:
    problem_summary = pd.DataFrame(columns=["instrument", "problem_type", "count"])

# =========================
# 10. 显示结果
# =========================

print("\nContent check finished.")
print("all samples    :", len(content_check))
print("valid samples  :", len(valid_samples))
print("invalid samples:", len(invalid_samples))

print("\nContent summary:")
display(content_summary)

print("\nProblem summary:")
display(problem_summary)

print("\nInvalid samples head:")
display(invalid_samples.head())

print("\nValid samples head:")
display(valid_samples.head())

# =========================
# 11. 保存结果
# =========================

content_check.to_csv(CONTENT_CHECK_PATH, index=False, encoding="utf-8-sig")
valid_samples.to_csv(VALID_SAMPLES_PATH, index=False, encoding="utf-8-sig")
invalid_samples.to_csv(INVALID_SAMPLES_PATH, index=False, encoding="utf-8-sig")
content_summary.to_csv(CONTENT_SUMMARY_PATH, index=False, encoding="utf-8-sig")
problem_summary.to_csv(PROBLEM_SUMMARY_PATH, index=False, encoding="utf-8-sig")

print("\nSaved files:")
print(CONTENT_CHECK_PATH)
print(VALID_SAMPLES_PATH)
print(INVALID_SAMPLES_PATH)
print(CONTENT_SUMMARY_PATH)
print(PROBLEM_SUMMARY_PATH)

# =========================
# 12. 保存后复查
# =========================

check_content = pd.read_csv(CONTENT_CHECK_PATH)
check_valid = pd.read_csv(VALID_SAMPLES_PATH)
check_invalid = pd.read_csv(INVALID_SAMPLES_PATH)

print("\nReload check:")
print("content_check.csv shape :", check_content.shape)
print("valid_samples.csv shape :", check_valid.shape)
print("invalid_samples.csv shape:", check_invalid.shape)

PROCESSED_DIR     : D:\organ-amt-generalization\data\processed\ex01_data_check
PAIRED_FILES_PATH : D:\organ-amt-generalization\data\processed\ex01_data_check\paired_files.csv

paired_files shape: (25, 19)


,piece_id,instrument,source,domain,audio_path,midi_path,has_audio,has_midi,metadata_duration,sample_rate,channels,split,composer,title,soundfont,synthesizer,maestro_year,maestro_audio_filename,maestro_midi_filename
0,faf_gm11,organ,ex01_bachorgan,organ,D:\organ-amt-generalization\data\raw\organ\ex0...,D:\organ-amt-generalization\data\raw\organ\ex0...,True,True,350.064036,44100.0,2.0,NaN,NaN,NaN,data/raw/soundfonts/Pipe Organ Samples v1.1.sf2,fluidsynth,NaN,NaN,NaN
1,faf_gm12,organ,ex01_bachorgan,organ,D:\organ-amt-generalization\data\raw\organ\ex0...,D:\organ-amt-generalization\data\raw\organ\ex0...,True,True,399.139410,44100.0,2.0,NaN,NaN,NaN,data/raw/soundfonts/Pipe Organ Samples v1.1.sf2,fluidsynth,NaN,NaN,NaN
2,fafxgm11,organ,ex01_bachorgan,organ,D:\organ-amt-generalization\data\raw\organ\ex0...,D:\organ-amt-generalization\data\raw\organ\ex0...,True,True,350.064036,44100.0,2.0,NaN,NaN,NaN,data/raw/soundfonts/Pipe Organ Samples v1.1.sf2,fluidsynth,NaN,NaN,NaN
3,fafxgm12,organ,ex01_bachorgan,organ,D:\organ-amt-generalization\data\raw\organ\ex0...,D:\organ-amt-generalization\data\raw\organ\ex0...,True,True,406.639456,44100.0,2.0,NaN,NaN,NaN,data/raw/soundfonts/Pipe Organ Samples v1.1.sf2,fluidsynth,NaN,NaN,NaN
4,fai_hm11,organ,ex01_bachorgan,organ,D:\organ-amt-generalization\data\raw\organ\ex0...,D:\organ-amt-generalization\data\raw\organ\ex0...,True,True,94.946395,44100.0,2.0,NaN,NaN,NaN,data/raw/soundfonts/Pipe Organ Samples v1.1.sf2,fluidsynth,NaN,NaN,NaN



Check thresholds:
SILENT_RMS_THRESHOLD   : 0.0001
CLIP_AMPLITUDE_THRESHOLD: 0.999
CLIP_RATIO_THRESHOLD   : 0.001
DURATION_DIFF_THRESHOLD: 3.0 sec


d:\anaconda3\envs\m1\Lib\site-packages\pretty_midi\pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(


checked 20/25
checked 25/25

Content check finished.
all samples    : 25
valid samples  : 25
invalid samples: 0

Content summary:


...


Invalid samples head:


,piece_id,instrument,source,domain,audio_path,midi_path,has_audio,has_midi,metadata_duration,sample_rate,...,min_pitch,max_pitch,mean_pitch,num_instruments,num_nonempty_instruments,num_programs,duration_diff_sec,duration_match,is_valid,problem_type



Valid samples head:


,piece_id,instrument,source,domain,audio_path,midi_path,has_audio,has_midi,metadata_duration,sample_rate,...,min_pitch,max_pitch,mean_pitch,num_instruments,num_nonempty_instruments,num_programs,duration_diff_sec,duration_match,is_valid,problem_type
0,faf_gm11,organ,ex01_bachorgan,organ,D:\organ-amt-generalization\data\raw\organ\ex0...,D:\organ-amt-generalization\data\raw\organ\ex0...,True,True,350.064036,44100.0,...,24.0,86.0,62.727469,4,4,1,2.436625,True,True,valid
1,faf_gm12,organ,ex01_bachorgan,organ,D:\organ-amt-generalization\data\raw\organ\ex0...,D:\organ-amt-generalization\data\raw\organ\ex0...,True,True,399.139410,44100.0,...,24.0,84.0,61.869333,4,4,1,2.514491,True,True,valid
2,fafxgm11,organ,ex01_bachorgan,organ,D:\organ-amt-generalization\data\raw\organ\ex0...,D:\organ-amt-generalization\data\raw\organ\ex0...,True,True,350.064036,44100.0,...,24.0,86.0,62.727469,7,7,2,2.436625,True,True,valid
3,fafxgm12,organ,ex01_bachorgan,organ,D:\organ-amt-generalization\data\raw\organ\ex0...,D:\organ-amt-generalization\data\raw\organ\ex0...,True,True,406.639456,44100.0,...,24.0,84.0,61.870853,7,7,2,2.514536,True,True,valid
4,fai_hm11,organ,ex01_bachorgan,organ,D:\organ-amt-generalization\data\raw\organ\ex0...,D:\organ-amt-generalization\data\raw\organ\ex0...,True,True,94.946395,44100.0,...,23.0,83.0,63.638743,3,3,1,2.453698,True,True,valid



Saved files:
D:\organ-amt-generalization\data\processed\ex01_data_check\content_check.csv
D:\organ-amt-generalization\data\processed\ex01_data_check\valid_samples.csv
D:\organ-amt-generalization\data\processed\ex01_data_check\invalid_samples.csv
D:\organ-amt-generalization\data\processed\ex01_data_check\content_check_summary.csv
D:\organ-amt-generalization\data\processed\ex01_data_check\problem_summary.csv

Reload check:
content_check.csv shape : (25, 44)
valid_samples.csv shape : (25, 44)
invalid_samples.csv shape: (0, 44)


In [9]:


PROJECT_ROOT = Path(r"D:\organ-amt-generalization")
PROCESSED_DIR = PROJECT_ROOT / r"data\processed\ex01_data_check"

INVALID_SAMPLES_PATH = PROCESSED_DIR / "invalid_samples.csv"

invalid_samples = pd.read_csv(INVALID_SAMPLES_PATH)

duration_mismatch = invalid_samples[
    invalid_samples["problem_type"].str.contains("duration_mismatch", na=False)
].copy()

cols = [
    "piece_id",
    "instrument",
    "audio_duration_sec",
    "midi_duration_sec",
    "duration_diff_sec",
    "num_notes",
    "audio_path",
    "midi_path",
]

duration_mismatch = duration_mismatch[cols].sort_values(
    "duration_diff_sec",
    ascending=False,
)

display(duration_mismatch)

print("duration mismatch count:", len(duration_mismatch))
print("min diff:", duration_mismatch["duration_diff_sec"].min())
print("mean diff:", duration_mismatch["duration_diff_sec"].mean())
print("max diff:", duration_mismatch["duration_diff_sec"].max())

duration_mismatch.to_csv(
    PROCESSED_DIR / "duration_mismatch_detail.csv",
    index=False,
    encoding="utf-8-sig",
)

print("saved:", PROCESSED_DIR / "duration_mismatch_detail.csv")

,piece_id,instrument,audio_duration_sec,midi_duration_sec,duration_diff_sec,num_notes,audio_path,midi_path


duration mismatch count: 0
min diff: nan
mean diff: nan
max diff: nan
saved: D:\organ-amt-generalization\data\processed\ex01_data_check\duration_mismatch_detail.csv
